# Kaggle - Vietnamese Medical NER: Train + Inference

1 notebook duy nhat: fine-tune LoRA (BUOC 1) + chay inference + tao submit (BUOC 2).

**Thu tu thuc hien tren Kaggle:**
1. Cell 1-2: Cai dat + kiem tra GPU
2. Cell 3: Clone repo GitHub
3. Cell 4: Cau hinh (MODEL + INFERENCE_MODE)
4. Cell 5: Train LoRA (neu TRAIN = True)
5. Cell 6: Kiem tra adapter
6. Cell 7: Load model + adapter
7. Cell 8: Chay inference tren test set
8. Cell 9: Tao file submit .zip

## Cell 1 - Cai dat thu vien

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
!pip install -q editdistance

## Cell 2 - Kiem tra GPU

In [ ]:
import torch, os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {gb:.1f} GB")

## Cell 3 - Lay code (3 cach)

Chon 1 trong 3 cach. KHUYEN NGHI cach 1 (Kaggle Dataset) hoac cach 2 (upload zip).

In [ ]:
import os, sys, shutil, zipfile

WORK_DIR = "/kaggle/working"
REPO_NAME = "Viettel-AI-Race"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

# ===== CHON CACH LAY CODE ============================
# METHOD = 0 → git clone (can public repo + internet)
# METHOD = 1 → Kaggle Dataset (kaggle datasets add data bphongcyrus/vtai-code)
# METHOD = 2 → Upload zip len Kaggle Datasets roi extract
METHOD = 1
# =====================================================

def via_git():
    GITHUB_USER = "bphong-cyrus"
    url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
    print(f"Cloning {url}...")
    r = os.system(f"cd {WORK_DIR} && git clone {url} 2>&1")
    if r != 0:
        raise RuntimeError("git clone that bai. Thu METHOD=1 hoac METHOD=2.")

def via_kaggle_dataset(dataset_name):
    src = f"/kaggle/input/{dataset_name}"
    if not os.path.isdir(src):
        raise RuntimeError(f"Khong thay {src}. Kiem tra ten dataset o Kaggle sidebar.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    shutil.copytree(src, REPO_DIR)
    print(f"Copied tu {src} -> {REPO_DIR}")

def via_zip_upload(zip_basename):
    zip_path = f"/kaggle/input/{zip_basename}"
    if not os.path.exists(zip_path):
        raise RuntimeError(f"Khong thay {zip_path}")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    os.makedirs(REPO_DIR)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f"Extracted {zip_path} -> {REPO_DIR}")

if os.path.exists(REPO_DIR):
    print(f"Repo da ton tai: {REPO_DIR}")
else:
    if METHOD == 0:
        via_git()
    elif METHOD == 1:
        DATASET_NAME = "vtai-code"   # sua ten dataset o day
        via_kaggle_dataset(DATASET_NAME)
    elif METHOD == 2:
        ZIP_BASENAME = "vtai-code.zip"   # sua ten file zip o day
        via_zip_upload(ZIP_BASENAME)
    else:
        raise ValueError("METHOD phai la 0, 1, hoac 2")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"\nWorking: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))[:20]}")

# Verify cac file can thiet
needed = ['llm_ner_config.py', 'llm_prompts.py', 'llm_inference.py',
          'hybrid_pipeline.py', 'sft_train_lora.py', 'v20_pipeline.py',
          'drug_dict_v3.py', 'vocab_v5.py', 'scorer.py']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print(f"\nTHIEU file: {missing}")
    print("Cach fix:")
    print("  - METHOD=1: kiem tra ten dataset o Kaggle sidebar ben phai")
    print("  - METHOD=2: kiem tra file zip co extract ra cac file .py khong")
    print("  - METHOD=0: can repo public + internet access")
else:
    print("\nTat ca file can thiet OK")

## Cell 4 - Cau hinh

DIEU CHINH CAC THAM SO O DAY THEO NHU CAU.

**Khi TRAIN = True:** cell 5 se chay training. 
**Khi TRAIN = False:** bo qua training, chay thang inference.

In [ ]:
# ===== CAU HINH ================================

# --- PHAN 1: CHON MODEL ---
# Base model - chon 1 trong 3:
#   "Qwen/Qwen2.5-7B-Instruct"       → JSON format tot, recommended
#   "Viet-Mistral/Vistral-7B-chat"   → tieng Viet tot nhat
#   "Qwen/Qwen2.5-3B-Instruct"       → nhanh, fit 6GB VRAM
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"


# --- PHAN 2: TRAIN HAY KHONG? ---
# TRAIN = True   -> chay Cell 5 de fine-tune LoRA
# TRAIN = False  -> bo qua training, dung model hien co
TRAIN = True

# LoRA params (chi can khi TRAIN = True)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 5
LR = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8          # effective batch = 8
MAX_SEQ_LEN = 4096
WARMUP_RATIO = 0.05
USE_QLORA = True        # 4-bit - bat buoc voi T4

# Duong dan LoRA da train (neu TRAIN = False)
# Dat = None neu muon dung base model (zero-shot)
LORA_ADAPTER = "/kaggle/working/lora_adapter"


# --- PHAN 3: INFERENCE ---
# Thu muc chua test set INPUT
#   - Neu test set nam trong repo: "input/input"
#   - Neu upload rieng len Kaggle: danh sach thu muc
TEST_INPUT_DIR = "input/input"

# Pipeline mode:
#   HYBRID = True  -> rules (v20) + LLM (tot nhat)
#   HYBRID = False -> LLM thuan
HYBRID = True
RULE_WEIGHT = 0.6        # Do tin cua rules trong hybrid (0.5 = trung binh)

# Output
OUTPUT_DIR = "/kaggle/working/output_hybrid"
ZIP_PATH = "/kaggle/working/output_hybrid.zip"


# ===== INAMG =====
print(f"Model      : {BASE_MODEL}")
print(f"Train LoRA : {TRAIN}")
if TRAIN:
    print(f"  LoRA     : r={LORA_R}, alpha={LORA_ALPHA}, qlora={USE_QLORA}")
    print(f"  Epochs   : {EPOCHS}, lr={LR}, bs={BATCH_SIZE}x{GRAD_ACCUM}")
else:
    print(f"  LoRA     : {LORA_ADAPTER}")
print(f"Hybrid    : {HYBRID} (rule_weight={RULE_WEIGHT})")
print(f"Test input : {TEST_INPUT_DIR}")
print(f"Output    : {OUTPUT_DIR}")

## Cell 5 - Train LoRA (chi chay khi TRAIN = True)

In [ ]:
if not TRAIN:
    print("TRAIN = False -> bo qua training.")
    print("Chuyen sang Cell 7 de load model.")
else:
    import argparse
    from sft_train_lora import main as train_main
    
    OUTPUT_DIR = "/kaggle/working/lora_adapter"
    
    args = argparse.Namespace(
        base_model=BASE_MODEL,
        gt_dir="bootstrap_gt",
        input_dir="input/input",
        output_dir=OUTPUT_DIR,
        epochs=EPOCHS,
        lr=LR,
        lora_r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        max_seq_len=MAX_SEQ_LEN,
        batch_size=BATCH_SIZE,
        grad_accum=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        qlora=USE_QLORA,
        gradient_checkpointing=True,
    )
    
    print("=" * 60)
    print("BAT DAU TRAINING...")
    print("=" * 60)
    
    train_main(args)
    
    print("=" * 60)
    print("TRAINING HOAN TAT!")
    print("=" * 60)

## Cell 6 - Kiem tra adapter (chi khi TRAIN = True)

In [ ]:
import os

print(f"=== Adapter files in {OUTPUT_DIR} ===")
if os.path.exists(OUTPUT_DIR):
    total = 0
    for f in sorted(os.listdir(OUTPUT_DIR)):
        path = f"{OUTPUT_DIR}/{f}"
        sz = os.path.getsize(path) / 1024 / 1024
        total += sz
        print(f"  {f}: {sz:.2f} MB")
    print(f"\n  Tong: {total:.1f} MB")
else:
    print("  WARNING: Adapter not found!")

## Cell 7 - Load model + LoRA adapter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# 4-bit quantization (bat buoc voi T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

adapter_to_use = LORA_ADAPTER if LORA_ADAPTER and os.path.exists(LORA_ADAPTER) else None
if adapter_to_use:
    print(f"Loading LoRA adapter: {adapter_to_use}")
    model = PeftModel.from_pretrained(model, adapter_to_use)
    print("LoRA adapter loaded")
else:
    print("WARNING: No LoRA adapter - using base model (zero-shot)")

model.eval()
print("Model ready for inference")

## Cell 8 - Chay Inference tren test set

Xu ly tat ca file .txt trong TEST_INPUT_DIR, xuat .json ra OUTPUT_DIR.

In [ ]:
import json, time, re, os
from pathlib import Path
import torch

import v20_pipeline
from hybrid_pipeline import merge_rule_and_llm
from llm_prompts import SYSTEM_PROMPT

MAX_NEW_TOKENS = 1500


def generate_with_model(text):
    """Mot LLM call, tra ve raw text."""
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Trich xuat thuc the y te tu:\n{text}\n\nJSON:"},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def parse_llm_output(raw_output):
    """Parse JSON tu LLM raw output."""
    patterns = [
        r'\[\s*\{[\s\S]*?\}\s*\]',
        r'\{[\s\S]*"entities"[\s\S]*\}',
    ]
    for pat in patterns:
        m = re.search(pat, raw_output)
        if m:
            try:
                parsed = json.loads(m.group())
                if isinstance(parsed, dict) and "entities" in parsed:
                    parsed = parsed["entities"]
                if isinstance(parsed, list):
                    return parsed
            except:
                continue
    return []


def validate_and_fix(entity, text):
    """Kiem tra position, tu dong fix neu can."""
    e = dict(entity)
    if "text" not in e or "type" not in e:
        return None
    etext = e["text"]
    if "start" in e and "end" in e and text[e["start"]:e["end"]] == etext:
        return e
    idx = text.find(etext)
    if idx >= 0:
        e["start"] = idx
        e["end"] = idx + len(etext)
    return e


def clean_entities(entities, text):
    """Loai bo hallucination, fix positions."""
    VALID_TYPES = {"THUOC", "TRIEU_CHUNG", "CHAN_DOAN"}
    STOPWORDS = {"khong", "khong co", "neu", "nao", "nay", "benh nhan", "nam", "nu", "tuoi"}
    result = []
    seen = set()
    for e in entities:
        if e.get("type") not in VALID_TYPES:
            continue
        etext = e.get("text", "").strip()
        if len(etext) < 2:
            continue
        if etext.lower() in STOPWORDS:
            continue
        e2 = validate_and_fix(e, text)
        if e2 is None:
            continue
        key = (e2["type"], e2["start"], e2["end"])
        if key not in seen:
            seen.add(key)
            result.append(e2)
    return result


def extract_llm(text):
    """LLM-only extraction."""
    raw = generate_with_model(text[:1800])
    entities = parse_llm_output(raw)
    return clean_entities(entities, text)


def hybrid_extract(text):
    """Hybrid: rules (v20) + LLM."""
    rule_ents = v20_pipeline.extract_entities_v20(text)
    llm_ents = extract_llm(text)
    return merge_rule_and_llm(rule_ents, llm_ents, rule_weight=RULE_WEIGHT)


extractor = hybrid_extract if HYBRID else extract_llm
mode_label = "HYBRID" if HYBRID else "LLM-only"
print(f"Mode: {mode_label} (rule_weight={RULE_WEIGHT})")

os.makedirs(OUTPUT_DIR, exist_ok=True)
test_files = sorted(Path(TEST_INPUT_DIR).glob("*.txt"),
                     key=lambda p: int(p.stem) if p.stem.isdigit() else 999)

print(f"\nProcessing {len(test_files)} files...")
t0 = time.time()
total_ents = 0
type_counts = {"THUOC": 0, "TRIEU_CHUNG": 0, "CHAN_DOAN": 0}

for i, fpath in enumerate(test_files, 1):
    text = fpath.read_text(encoding="utf-8")
    ents = extractor(text)
    
    with open(f"{OUTPUT_DIR}/{fpath.stem}.json", "w", encoding="utf-8") as f:
        json.dump(ents, f, ensure_ascii=False, indent=2)
    
    total_ents += len(ents)
    for e in ents:
        type_counts[e.get("type", "?")] = type_counts.get(e.get("type", "?"), 0) + 1
    
    if i % 20 == 0 or i == len(test_files):
        elapsed = time.time() - t0
        eta = elapsed / i * (len(test_files) - i)
        print(f"  [{i}/{len(test_files)}] {elapsed:.0f}s | ETA {eta:.0f}s | ents={total_ents}")

print(f"\nHoan tat! {len(test_files)} files")
print(f"Total entities: {total_ents}")
print(f"By type: {type_counts}")
print(f"Time: {time.time()-t0:.0f}s")
print(f"Output: {OUTPUT_DIR}")

## Cell 9 - Tao file submit .zip

In [ ]:
import zipfile, os
from pathlib import Path

print(f"Creating {ZIP_PATH}...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(Path(OUTPUT_DIR).glob("*.json")):
        zf.write(f, f.name)

zip_size = os.path.getsize(ZIP_PATH) / 1024
n_files = len(list(Path(OUTPUT_DIR).glob("*.json")))

print(f"ZIP: {ZIP_PATH}")
print(f"Files: {n_files}, Size: {zip_size:.1f} KB")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = sorted(zf.namelist())
    print(f"Verified: {len(names)} entries")
    print(f"Sample: {names[:5]}")

print("\n=== SAN SANG SUBMIT! ===")
print(f"Download: {ZIP_PATH}")